# LangChain: Evaluation

## Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation
* LangChain evaluation platform

本节讲的是如何评估一个基于 LLM 的问答系统（如上一节的 RetrievalQA）：
1. 先手动/自动构造一批"问题-标准答案"样例（examples）
2. 让 QA 系统对这些问题生成预测答案
3. 用另一个 LLM（LLM-assisted evaluation）自动判断预测答案是否正确，而不是靠人工逐条核对

> 注：本 notebook 依赖的 `OutdoorClothingCatalog_1000.csv` 数据文件在当前目录下不存在，
> 涉及读取该文件的 cell 无法在本地完整跑通，这是数据缺失问题，不是代码逻辑 bug。
> 已修复所有 import 路径、`VectorstoreIndexCreator` 缺少 embedding 参数、以及 `langchain.debug` 失效等版本兼容性问题。

In [ ]:
# 加载 .env 中的环境变量（如 OPENAI_API_KEY）
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [ ]:
# account for deprecation of LLM model
# 根据当前日期判断该用哪个 gpt-3.5-turbo 版本名，逻辑本身没问题
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

In [ ]:
# 【版本兼容性修复】同 L4，正确的导入路径：
#   RetrievalQA / VectorstoreIndexCreator -> langchain_classic
#   ChatOpenAI                             -> langchain_openai
#   CSVLoader / DocArrayInMemorySearch     -> langchain_community
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import CSVLoader
from langchain_classic.indexes import VectorstoreIndexCreator
from langchain_community.vectorstores import DocArrayInMemorySearch

In [ ]:
# 加载产品目录 CSV 并解析成 Document 列表
# 【环境限制】OutdoorClothingCatalog_1000.csv 在当前目录下不存在，这里无法实际运行，不是代码 bug
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

In [ ]:
# 【真实 bug 修复】同 L4 的问题：VectorstoreIndexCreator 在当前版本中 embedding 参数是必填的，
# 不传会报 pydantic ValidationError（已在本地用 venv 验证）。这里补上 embedding=OpenAIEmbeddings()。
from langchain_openai import OpenAIEmbeddings

index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=OpenAIEmbeddings(),
).from_loaders([loader])

In [ ]:
# 构建待评估的问答系统：基于产品目录的 RetrievalQA
# document_separator 自定义了多条检索结果拼接时的分隔符，方便在 verbose 输出里区分不同文档
llm = ChatOpenAI(temperature = 0.0, model=llm_model)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.vectorstore.as_retriever(),
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

In [ ]:
# 看看第 11 条数据长什么样，方便手动构造评估样例（examples）时参考
data[10]

In [ ]:
data[11]

In [ ]:
# 人工手写两条评估样例：{"query": 问题, "answer": 标准答案}
# 这是"人工评估"（Manual evaluation）的起点——先靠人工写一些已知答案的问题
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

In [ ]:
# 【版本兼容性修复】原代码 from langchain.evaluation.qa import QAGenerateChain 已不存在
# （langchain.evaluation 模块在当前版本中不存在），正确路径是 langchain_classic.evaluation.qa。
# QAGenerateChain：用 LLM 根据一段文档自动生成"问题-答案"对，比纯人工编写样例更省力、能覆盖更多文档
from langchain_classic.evaluation.qa import QAGenerateChain


In [ ]:
# 用一个 LLM 驱动的 chain 来自动生成问答样例
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(model=llm_model))

In [ ]:
# 为前 5 条文档分别生成一个问答样例
# 【版本兼容性提示】apply_and_parse() 在当前版本会打印一条 UserWarning：
# "The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain."
# 但功能仍然正常（已在本地用 venv 验证：调用本身不报错，只是打印警告），这里保留原写法。
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

In [ ]:
# 查看自动生成的第一条样例，格式应该跟手写的 examples 一致：{"query": ..., "answer": ...}
new_examples[0]

In [ ]:
# 对比一下生成样例所依据的原始文档内容
data[0]

In [ ]:
# 把人工样例和自动生成样例合并成一个完整的评估集
examples += new_examples

In [ ]:
# 【提示】.run() 是旧式调用方法，当前版本仍可用（deprecated），新写法是 qa.invoke({"query": ...})
qa.run(examples[0]["query"])

In [ ]:
# 【真实 bug 修复】原代码 langchain.debug = True 在当前版本中是一个"死代码"：
# langchain 顶层包已经没有 debug 这个开关属性了（dir(langchain) 里搜不到 'debug'），
# 直接赋值只是给 langchain 模块随手挂了一个没人读取的普通属性，完全不会真正打开调试日志
# （已在本地用 venv 验证：设置后 langchain_core.globals.get_debug() 仍然是 False）。
# 真正生效的开关在 langchain_core.globals 里，需要调用 set_debug(True)。
from langchain_core.globals import set_debug
set_debug(True)

In [ ]:
# 开启 debug 后再跑一次，应该能看到详细的中间步骤日志（prompt、检索到的文档等）
qa.run(examples[0]["query"])

In [ ]:
# Turn off the debug mode
# 同上，用 set_debug(False) 才能真正关闭调试日志
set_debug(False)

In [ ]:
# .apply() 会对 examples 里的每一条依次调用 qa，得到预测结果列表
# 【提示】.apply() 也是旧式方法，当前版本仍可用（deprecated）
predictions = qa.apply(examples)

In [ ]:
# 【版本兼容性修复】同前，正确路径是 langchain_classic.evaluation.qa
# QAEvalChain：用另一个 LLM 判断"预测答案"和"标准答案"是否语义一致（而不是要求逐字相同）
from langchain_classic.evaluation.qa import QAEvalChain

In [ ]:
llm = ChatOpenAI(temperature=0, model=llm_model)
eval_chain = QAEvalChain.from_llm(llm)

In [ ]:
# evaluate(examples, predictions)：把"标准答案"和"预测答案"成对喂给评估 LLM，逐条打分（CORRECT/INCORRECT）
graded_outputs = eval_chain.evaluate(examples, predictions)

In [ ]:
# 逐条打印问题、标准答案、预测答案、评估打分，方便人工复核
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i]['text'])
    print()

In [ ]:
# 查看第一条打分结果的原始结构
graded_outputs[0]